# GravityOCR in 10 minutes

Serve [`trillionlabs/GravityOCR`](https://huggingface.co/trillionlabs/GravityOCR), read text, a table
and a formula from image crops, and compare self-speculative decoding with plain autoregressive
decoding. The notebook picks its backend for the machine it runs on:

| machine | `BACKEND` | server |
|---|---|---|
| Linux with an NVIDIA L40 or H100 | `sglang` | `serve.sh`: the tutorial environment |
| Apple Silicon Mac | `mac` | `serve_mac.py`, on the [Gravity](https://gravityocr.trillionlabs.co) app's engine |

Do the one-time setup below, then run all cells: about 5–10 minutes. Settings, details and
troubleshooting are in [`README.md`](README.md#tutorial-notebook).

> **With Claude Code or Codex:** ask *"Do the one-time setup in
> inference/gravityocr/getting_started.ipynb, run the notebook headless, and summarize the PASS lines
> and the comparison table."* Sandbox settings: [README → With a coding
> agent](README.md#with-a-coding-agent).

## Before you start (once)

- **Notebook kernel:** any Python ≥ 3.10. The cell below installs Pillow and matplotlib into it
  if they are missing.
- **Linux, L40 or H100:** a CUDA 12.x toolkit, `git` and [`uv`](https://docs.astral.sh/uv/). Build the
  server environment (several GB, not counted in the 10 minutes):
  ```bash
  SGLANG_SRC=$HOME/src/sglang-gravityocr bash inference/gravityocr/sglang/setup.sh
  ```
- **Mac:** install [Gravity](https://github.com/trillion-labs/GravityOCR-macOS/releases) into
  Applications (no need to open it). The first run downloads the model once (2.2 GB).

In [ ]:
# Kernel requirements: Python >= 3.10, Pillow (renders the test crops) and matplotlib (the formula
# crop and the plot). Missing packages are installed into this kernel, with uv if there is one.
import importlib
import importlib.util
import pathlib
import shutil
import subprocess
import sys

REQUIREMENTS = {"PIL": "pillow", "matplotlib": "matplotlib"}  # module -> package
assert sys.version_info >= (3, 10), \
    f"this kernel is Python {sys.version.split()[0]}: pick a Python >= 3.10 kernel"
missing = [pkg for mod, pkg in REQUIREMENTS.items() if importlib.util.find_spec(mod) is None]
if missing:
    uv = shutil.which("uv") or next(
        (str(p) for p in (pathlib.Path.home() / ".local/bin/uv", pathlib.Path("/opt/homebrew/bin/uv"))
         if p.exists()), None)
    cmd = ([uv, "pip", "install", "--python", sys.executable, *missing] if uv
           else [sys.executable, "-m", "pip", "install", *missing])
    print(f"installing {', '.join(missing)} into {sys.executable}")
    if subprocess.run(cmd).returncode != 0:
        raise RuntimeError(f"could not install {missing}: run `uv pip install --python "
                           f"{sys.executable} {' '.join(missing)}`, then re-run this cell")
    importlib.invalidate_caches()
import matplotlib  # noqa: E402,F401  (fails here, not mid-tutorial, if the install did not work)
import PIL  # noqa: E402,F401

print(f"kernel OK: Python {sys.version.split()[0]} at {sys.executable}")

## 1. Start the server

Settings come from environment variables (for example `BACKEND=mac`, `GPU=1`); the defaults suit most
machines.

In [ ]:
import os
import pathlib

BACKEND = os.environ.get("BACKEND", "auto")  # "auto" | "sglang" (Linux, L40 / H100) | "mac"
GRAVITY_APP = os.environ.get("GRAVITY_APP", "/Applications/Gravity.app")  # BACKEND=mac
SGLANG_SRC = pathlib.Path(os.environ.get("SGLANG_SRC", "~/src/sglang-gravityocr")).expanduser()
VBIN = os.environ.get("VBIN") or str(SGLANG_SRC / ".venv-sglang/bin")  # setup.sh prints it
CKPT = os.environ.get("GRAVITYOCR_MODEL", "trillionlabs/GravityOCR")   # Hub id or local directory
GPU = os.environ.get("GPU", "auto")             # "auto", or a CUDA device index such as "0"
CAUSAL_GPU = os.environ.get("CAUSAL_GPU", "auto")  # "auto": a second free GPU if there is one
COMPARE_CAUSAL = os.environ.get("COMPARE_CAUSAL", "1") == "1"  # section 5
PORT = int(os.environ.get("PORT", "30000"))
CAUSAL_PORT = int(os.environ.get("CAUSAL_PORT", "30001"))
CUDA_HOME = os.environ.get("CUDA_HOME", "/usr/local/cuda-12.8")  # a CUDA 12.x toolkit (nvcc)
API_KEY = os.environ.get("API_KEY", "")  # optional; export it, never paste a key into a cell

Helpers: start, wait, stop, one OCR request. Run the cell; expand it to read the code.

In [ ]:
import atexit
import json
import platform
import signal
import socket
import subprocess
import sys
import tempfile
import time
import urllib.error
import urllib.request

T0 = time.time()
HERE = pathlib.Path.cwd()
if not (HERE / "serve.sh").exists():
    HERE = HERE / "inference/gravityocr"
assert (HERE / "serve.sh").exists(), "start Jupyter in inference/gravityocr/ or the repo root"
if BACKEND == "auto":
    BACKEND = "mac" if sys.platform == "darwin" and platform.machine() == "arm64" else "sglang"
GRAVITY_PY = pathlib.Path(GRAVITY_APP, "Contents/Resources/runtime/bin/python")
if BACKEND == "mac":
    assert GRAVITY_PY.exists(), f"Gravity not found at {GRAVITY_APP}: see 'Before you start', option B"
else:
    assert pathlib.Path(VBIN, "python").exists(), \
        f"no server env at VBIN={VBIN}: run sglang/setup.sh first (see 'Before you start', option A)"
print(f"BACKEND = {BACKEND}")
sys.path.insert(0, str(HERE))
import ocr_request  # noqa: E402  (the CLI client in this folder; we reuse its payload builder)

servers = {}  # port -> (Popen, log path)
SUPPORTED_CC = {"8.9": "Ada (L40)", "9.0": "Hopper (H100)"}


def list_gpus():
    """{index: (name, compute capability)} of supported GPUs, least memory in use first."""
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=index,name,compute_cap,memory.used",
                              "--format=csv,noheader,nounits"], capture_output=True, text=True,
                             timeout=60).stdout
    except (OSError, subprocess.SubprocessError):
        return {}
    rows = [[x.strip() for x in line.split(",")] for line in out.splitlines() if line.strip()]
    return {i: (name, cap) for i, name, cap, used in sorted(rows, key=lambda r: float(r[3]))
            if cap in SUPPORTED_CC}


def check_toolchain():
    """serve.sh's startup checks other than the GPU one: a CUDA 12.x nvcc and ninja in VBIN."""
    try:
        nvcc = subprocess.run([f"{CUDA_HOME}/bin/nvcc", "--version"], capture_output=True,
                              text=True, timeout=60).stdout
    except OSError:
        nvcc = ""
    if "release 12." not in nvcc:
        raise RuntimeError(f"no CUDA 12.x nvcc at {CUDA_HOME}/bin: set CUDA_HOME")
    if not pathlib.Path(VBIN, "ninja").exists():
        raise RuntimeError(f"no ninja in VBIN={VBIN}: re-run sglang/setup.sh")


def launch(mode, gpu, port, **env):
    """Start a server (serve.sh, or serve_mac.py on a Mac) in its own process group.

    Its log goes to a private temp file: serve.sh's log records the API keys."""
    if port in servers and servers[port][0].poll() is None:
        print(f"MODE={mode} already running on port {port} (started earlier in this kernel)")
        return
    with socket.socket() as s:
        if s.connect_ex(("127.0.0.1", port)) == 0:
            raise RuntimeError(
                f"port {port} is in use, probably by a server from another notebook kernel: restart or "
                f"shut down that kernel, or set PORT / CAUSAL_PORT. See: lsof -iTCP:{port} -sTCP:LISTEN")
    if BACKEND == "mac":
        # -B: never write __pycache__ into Gravity.app, which would break the app's signature
        cmd = [str(GRAVITY_PY), "-I", "-B", str(HERE / "serve_mac.py"), "--mode", mode,
               "--port", str(port), "--model", CKPT]
        where = "the Mac GPU (Gravity's MLX engine)"
    else:
        cmd = ["bash", str(HERE / "serve.sh")]
        if GPUS.get(gpu, ("", "9.0"))[1] != "9.0":
            # serve.sh accepts only compute capability 9.0: run its other checks, skip the GPU one
            check_toolchain()
            env.setdefault("SKIP_PREFLIGHT", "1")
        env = {"VBIN": VBIN, "MODE": mode, "CKPT": CKPT, "GPU": gpu, "PORT": str(port),
               "CUDA_HOME": CUDA_HOME, "CUDA_DEVICE_ORDER": "PCI_BUS_ID", **env}  # nvidia-smi order
        where = f"GPU {gpu} ({GPUS.get(gpu, ('unknown GPU',))[0]})"
    log = pathlib.Path(tempfile.gettempdir()) / f"gravityocr-{mode}-{port}.log"
    fd = os.open(log, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
    proc = subprocess.Popen(cmd, stdout=fd, stderr=subprocess.STDOUT, env={**os.environ, **env},
                            start_new_session=True)
    os.close(fd)
    servers[port] = (proc, log)
    print(f"starting MODE={mode} on {where}, port {port}; log: {log}")


def stop(port):
    """Stop a server this notebook started (SIGTERM to its process group, SIGKILL after 60 s)."""
    proc, _ = servers.pop(port, (None, None))
    if proc is None or proc.poll() is not None:
        return
    os.killpg(proc.pid, signal.SIGTERM)
    try:
        proc.wait(timeout=60)
    except subprocess.TimeoutExpired:
        os.killpg(proc.pid, signal.SIGKILL)
        proc.wait()
    print(f"stopped the server on port {port}")


atexit.register(lambda: [stop(p) for p in list(servers)])  # also after a failed headless run


def tail(port, n=30):
    """The last n lines of a server's log."""
    return "\n".join(servers[port][1].read_text(errors="replace").splitlines()[-n:])


def _headers():
    return {"Authorization": f"Bearer {API_KEY}"} if API_KEY else {}


def wait_until_ready(port, timeout=1800):
    """Poll /v1/models until it answers; fail with the log tail if the server exits."""
    t0, last = time.time(), time.time()
    while time.time() - t0 < timeout:
        proc = servers[port][0]
        if proc.poll() is not None:
            raise RuntimeError(f"the server exited with status {proc.returncode}:\n{tail(port)}")
        try:
            req = urllib.request.Request(f"http://127.0.0.1:{port}/v1/models", headers=_headers())
            with urllib.request.urlopen(req, timeout=10) as r:
                model = json.loads(r.read())["data"][0]["id"]
            print(f"port {port} ready after {time.time() - t0:.0f} s (model {model!r})")
            return model
        except urllib.error.HTTPError as e:
            if e.code in (401, 403):
                raise RuntimeError("the server wants a key: export API_KEY") from None
        except (urllib.error.URLError, OSError, ValueError):
            pass
        if time.time() - last >= 60:
            print(f"  {time.time() - t0:4.0f} s  {tail(port, 1)[:140]}")
            last = time.time()
        time.sleep(3)
    raise TimeoutError(f"port {port} not ready after {timeout} s:\n{tail(port)}")


def ocr(image, prompt="Text Recognition:", port=PORT, max_tokens=4096):
    """One crop (PNG bytes) -> {"text", "tokens", "seconds"}: the payload ocr_request.py sends."""
    payload = ocr_request.build_payload(image, "image/png", prompt, "glm-ocr", max_tokens)
    req = urllib.request.Request(f"http://127.0.0.1:{port}/v1/chat/completions", method="POST",
                                 data=json.dumps(payload).encode(),
                                 headers={"Content-Type": "application/json", **_headers()})
    t = time.perf_counter()
    try:
        with urllib.request.urlopen(req, timeout=300) as r:
            out = json.loads(r.read())
    except urllib.error.HTTPError as e:
        raise RuntimeError(f"HTTP {e.code}: {e.read().decode('utf-8', 'replace')[:300]}") from None
    return {"text": out["choices"][0]["message"]["content"],
            "tokens": (out.get("usage") or {}).get("completion_tokens", 0),
            "seconds": time.perf_counter() - t}

In [ ]:
GPUS = list_gpus() if BACKEND == "sglang" else {}
if GPU == "auto":
    GPU = next(iter(GPUS), "0")
if CAUSAL_GPU == "auto":
    CAUSAL_GPU = next((g for g in GPUS if g != GPU), "")  # "" = take turns on GPU

launch("self-spec", GPU, PORT)
if COMPARE_CAUSAL and BACKEND == "sglang" and CAUSAL_GPU:
    launch("causal", CAUSAL_GPU, CAUSAL_PORT)  # warms up in parallel on a second GPU

## 2. How it works (read while the server starts)

GravityOCR is [GLM-OCR](https://huggingface.co/zai-org/GLM-OCR) fine-tuned so that one model, with a
single output head, decodes both ways: given a block of masks with bidirectional attention it fills the
whole block at once (diffusion), and with causal attention it predicts the next token (AR).
Self-speculative decoding alternates the two modes of that one model:

```
DRAFT   last token + 32 masks  ─► 32 proposed tokens in one bidirectional (diffusion) pass
VERIFY  one causal (AR) pass   ─► keep the longest prefix the AR predictions agree with, plus the next one
        ═► up to 34 tokens per round, each one the model's greedy AR choice
```

`MODE=causal` decodes the same weights one token per pass: the reference. The first start on an L40
compiles kernels, which takes a few minutes.

In [ ]:
wait_until_ready(PORT)
for line in servers[PORT][1].read_text(errors="replace").splitlines():
    # the mask id must come from the checkpoint's block_diffusion.json; on L40 the ViT graphs are OFF
    if any(tag in line for tag in ("[selfspec-diffusion]", "[ss-vit-graph]", "[serve_mac] mode=")):
        print(line.strip()[:200])

if COMPARE_CAUSAL and BACKEND == "mac":
    launch("causal", GPU, CAUSAL_PORT)  # the checkpoint is cached now, so this loads in seconds

## 3. First request

One request is one cropped region plus a task prompt, as an OpenAI chat completion:

```json
{"model": "glm-ocr", "temperature": 0, "max_tokens": 4096,
 "messages": [{"role": "user", "content": [
   {"type": "image_url", "image_url": {"url": "data:image/png;base64,..."}},
   {"type": "text", "text": "Text Recognition:"}]}]}
```

In [ ]:
from IPython.display import HTML, Image, Math, display

hello = ocr_request.render_text_png("Hello GravityOCR")
display(Image(hello))
r = ocr(hello)
print(repr(r["text"]), f"({r['tokens']} tokens, {r['seconds']:.2f} s)")
print("PASS: read back" if " ".join(r["text"].split()) == "Hello GravityOCR" else "CHECK: text differs")

## 4. Text, a table and a formula

`Text Recognition:` returns text, `Table Recognition:` HTML, `Formula Recognition:` LaTeX. Send one
region per request; add your own crops to `MY_CROPS`.

In [ ]:
import io

from PIL import Image as PILImage
from PIL import ImageDraw, ImageFont


def table_png(rows, w=170, h=46):
    """A ruled table with a grey header row."""
    img = PILImage.new("RGB", (w * len(rows[0]) + 2, h * len(rows) + 2), "white")
    d = ImageDraw.Draw(img)
    try:
        font = ImageFont.load_default(size=22)
    except TypeError:  # Pillow < 10.1
        font = ImageFont.load_default()
    for i, row in enumerate(rows):
        for j, cell in enumerate(row):
            box = (j * w, i * h, (j + 1) * w, (i + 1) * h)
            d.rectangle(box, outline="black", width=2, fill="#e8e8e8" if i == 0 else "white")
            d.text((box[0] + 12, box[1] + 11), cell, fill="black", font=font)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()


def formula_png(latex):
    """A display formula rendered by matplotlib mathtext (plain text without matplotlib)."""
    try:
        from matplotlib.figure import Figure  # not pyplot: leaves the notebook's backend alone
    except ImportError:
        return ocr_request.render_text_png(latex)
    fig = Figure(figsize=(0.01, 0.01))
    fig.text(0, 0, f"${latex}$", fontsize=30)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=100, bbox_inches="tight", pad_inches=0.25, facecolor="white")
    return buf.getvalue()


CROPS = {
    "paragraph": (ocr_request.render_text_png(
        "GravityOCR drafts a block of tokens by diffusion\nand verifies it autoregressively, with one\n"
        "model and a single output head.", font_size=28), "Text Recognition:"),
    "table": (table_png([["Model", "Mode", "Pass"], ["GravityOCR", "self-spec", "draft"],
                         ["GravityOCR", "causal", "verify"]]), "Table Recognition:"),
    "formula": (formula_png(r"\int_0^1 x^2\,dx = \frac{1}{3}"), "Formula Recognition:"),
}
MY_CROPS = {}  # {"name": ("path/to/crop.png", one of the three prompts)}
for name, (path, prompt) in MY_CROPS.items():
    CROPS[name] = (pathlib.Path(path).expanduser().read_bytes(), prompt)

for name, (img, prompt) in CROPS.items():
    print(f"── {name}  ({prompt})")
    display(Image(img))
    r = ocr(img, prompt)
    print(r["text"], f"\n[{r['tokens']} tokens, {r['seconds']:.2f} s]")
    if prompt.startswith("Table"):
        display(HTML(r["text"]))
    elif prompt.startswith("Formula"):
        display(Math(r["text"].strip().strip("$").strip()))

The same request from a shell:

In [ ]:
!"{sys.executable}" "{HERE}/ocr_request.py" --synthetic "Hello GravityOCR"

## 5. Self-spec vs causal

Expect mostly identical text (bf16 near-ties can differ) and the largest speedup on long outputs. These
are single requests after a warm-up: latency, not throughput. Mac numbers are not L40 numbers.

In [ ]:
BENCH = dict(CROPS)
BENCH["long paragraph"] = (ocr_request.render_text_png("\n".join(
    f"{i}. Self-speculative decoding commits up to thirty-four tokens per round."
    for i in range(1, 13)), font_size=24), "Text Recognition:")


def run_set(port):
    ocr(hello, port=port)  # warm-up
    return {name: ocr(img, prompt, port=port) for name, (img, prompt) in BENCH.items()}


res = {"self-spec": run_set(PORT)}
if COMPARE_CAUSAL:
    if CAUSAL_PORT not in servers:  # one GPU: stop self-spec, then start causal on the same GPU
        stop(PORT)
        launch("causal", GPU, CAUSAL_PORT)
    wait_until_ready(CAUSAL_PORT)
    res["causal"] = run_set(CAUSAL_PORT)

print(f"{'crop':<15}{'tokens':>7}{'self-spec s':>13}{'causal s':>10}{'speedup':>9}  same text")
for name in BENCH:
    s, c = res["self-spec"][name], res.get("causal", {}).get(name)
    print(f"{name:<15}{s['tokens']:>7}{s['seconds']:>13.3f}" + (
        f"{c['seconds']:>10.3f}{c['seconds'] / s['seconds']:>8.2f}x  "
        f"{'yes' if c['text'] == s['text'] else 'NO'}" if c else ""))
if "causal" in res:
    same = sum(res["causal"][n]["text"] == res["self-spec"][n]["text"] for n in BENCH)
    print(f"\nPASS: compared {len(BENCH)} crops, {same} identical")

In [ ]:
if "causal" in res:
    try:
        from matplotlib.figure import Figure
    except ImportError:
        Figure = None
    if Figure:
        names = list(BENCH)
        fig = Figure(figsize=(7, 2.6))
        ax = fig.subplots()
        x = range(len(names))
        ax.bar([i - 0.2 for i in x], [res["causal"][n]["seconds"] for n in names], 0.4,
               label="causal", color="#9AA5B1")
        ax.bar([i + 0.2 for i in x], [res["self-spec"][n]["seconds"] for n in names], 0.4,
               label="self-spec", color="#3D6FB6")
        ax.set_xticks(list(x), names)
        ax.set_ylabel("seconds per crop")
        ax.legend(frameon=False)
        ax.spines[["top", "right"]].set_visible(False)
        fig.tight_layout()
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=100)  # a PNG shows in any Jupyter and in headless runs
        display(Image(buf.getvalue()))

## 6. Clean up

In [ ]:
for port in list(servers):
    stop(port)
print(f"tutorial time: {(time.time() - T0) / 60:.1f} min (first cell to here, server starts included)")

**Next:** your own crops in `MY_CROPS`, more replicas with `DP=2 GPU=0,1 bash serve.sh`, or your own
checkpoint with `CKPT=/path` (check it with appendix B first).

---

# Appendix (optional)

## A. What the L40 recipe contains

Upstream SGLang pinned at v0.5.10, one patch, and four new files:

| new file | role |
|---|---|
| `selfspec_diffusion_worker.py` | the self-spec loop: one draft pass, one verify pass |
| `selfspec_fast_attn.py` | draft and verify attention without a custom mask |
| `selfspec_fast_commit.py` | a round's commit in one GPU kernel |
| `glm_vit_graph_runner.py` | the vision encoder as CUDA graphs (H100 only; off on an L40) |

In [ ]:
import re

sg = HERE / "sglang"
patch = (sg / "gravityocr_selfspec.patch").read_text()
touched = re.findall(r"^diff --git a/(\S+) b/", patch, flags=re.M)
lock = dict(re.findall(r"^([A-Za-z0-9._-]+)==(\S+)", (sg / "requirements-lock.txt").read_text(),
                       flags=re.M))
print(f"upstream sgl-project/sglang @ {(sg / 'BASE_COMMIT.txt').read_text()[:12]} (v0.5.10)")
print(f"patch: {len(touched)} files —", ", ".join(f.split("/")[-1] for f in touched))
for p in sorted((sg / "python").rglob("*.py")):
    print(f"new:   {p.relative_to(sg / 'python')} ({len(p.read_text().splitlines())} lines)")
print(f"lock:  {len(lock)} pins, among them", ", ".join(
    f"{k} {lock[k]}" for k in ("torch", "transformers", "flashinfer-python") if k in lock))

## B. Check a checkpoint

With a wrong `mask_id` the server still answers correctly, but self-spec gets no faster. The full
contract is in [`MODEL_CONTRACT.md`](MODEL_CONTRACT.md).

In [ ]:
def ckpt_json(name):
    if pathlib.Path(CKPT).expanduser().is_dir():
        return json.loads((pathlib.Path(CKPT).expanduser() / name).read_text())
    url = f"https://huggingface.co/{CKPT}/resolve/main/{name}"
    with urllib.request.urlopen(url, timeout=120) as r:
        return json.loads(r.read())


cfg, bd, tok = (ckpt_json(f) for f in ("config.json", "block_diffusion.json", "tokenizer.json"))
mask = {t["id"]: t["content"] for t in tok.get("added_tokens", [])}.get(bd.get("mask_id"))
arch, ar_w, bd_size = cfg.get("architectures"), bd.get("ar_loss_weight") or 0, bd.get("bd_size", 0)
for ok, what in [
    (arch == ["GlmOcrForConditionalGeneration"], f"architecture {arch}"),
    ("auto_map" not in cfg, "no auto_map (no remote code)"),
    (ar_w > 0, f"ar_loss_weight {ar_w} > 0 (trained to decode AR as well)"),
    (mask == "<|mask|>", f"mask_id {bd.get('mask_id')} is {mask!r} in tokenizer.json"),
    (bd_size + 1 == 33, f"bd_size {bd_size} -> DRAFT {bd_size + 1} (serve.sh default 33)"),
]:
    print("✓" if ok else "✗", what)

## C. Security

The servers listen on `127.0.0.1` only. Do not expose the port; before sharing a server, read
[README → Security](README.md#security).

## D. Troubleshooting

| symptom | fix |
|---|---|
| `no server env at VBIN=…` | run `sglang/setup.sh` (Before you start) |
| `Gravity not found at …` | install Gravity (Before you start) |
| `the server exited with status …` | the log tail above names the cause; `print(tail(PORT, 200))` shows more |
| `port 30000 is in use` | another kernel's servers are still up: shut that kernel down |
| `No Metal device available` | the notebook ran inside a sandbox: run it outside ([README](README.md#with-a-coding-agent)) |
| `no CUDA 12.x nvcc` | point `CUDA_HOME` at a CUDA 12.x toolkit |
| self-spec no faster than causal | run appendix B |

More: [README → Known limits](README.md#known-limits).